<a href="https://colab.research.google.com/github/sensiwise-ai/agentic-ai-workshop2026-labs/blob/main/Lab4_Agent_Team_Day2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: A team of agents: specialists, a reviewer, a supervisorn

Yesterday one agent held every tool. Today the same job is done by a **team**: a triage agent that decides who is needed, a policy specialist and an account specialist that each know one thing well, a writer, a **reviewer who is not the writer**, and a supervisor that fixes the order, enforces a budget and knows when to hand a case to a person.

This lab explores the benefits of using a team of specialized agents to handle customer service inquiries, contrasting it with a monolithic single-agent approach. Each agent has a distinct role and limited access to information, mimicking a real-world organizational structure.

```
message ─▶ triage ─▶ which specialists? ─▶ policy specialist (Lab 1: retrieval)  ─┐
                    │                       account specialist (Lab 2: SQL)         ├─▶ case file ─▶ writer ─▶ reviewer ──returned──▶ writer again, or a specialist again
                    └─ escalate now ─▶ a person                                     ┘                                └──approved──▶ the reply      ──refused twice──▶ a person
```

The diagram above illustrates the workflow of the agent team. Messages are first triaged, then routed to relevant specialists. Their findings contribute to a shared 'case file,' which the writer uses to draft a reply. A reviewer then checks the draft, potentially sending it back for revisions or escalating to a human. The supervisor oversees the entire process, managing the flow and enforcing rules.

Then the honest question: **is the team better than one agent with all the tools?** The same three cases go through both, judged by the same reviewer, and a table shows what each path cost and what it got right. That trade-off is the decision every capstone team makes this afternoon.

### Setup: Runtime → Change runtime type → **T4 GPU**, then run this cell

This step is crucial for running the language models efficiently. A T4 GPU provides the necessary computational power and memory for loading and running the `Qwen/Qwen3.5-4B` model, which can be memory-intensive. Ensure you select 'T4 GPU' under the 'Runtime' -> 'Change runtime type' menu before executing the setup cells.

In [ ]:
# Install necessary libraries for the lab. These include 'transformers' for language models,
# 'accelerate' for optimized model loading, 'chromadb' for vector database functionalities,
# 'sentence-transformers' for embedding generation, and 'pypdf' for PDF processing.
!pip install -q -U transformers accelerate chromadb sentence-transformers pypdf

# Import required Python modules.
import torch, json, re
from transformers import AutoTokenizer, AutoModelForImageTextToText

# Define the ID of the pre-trained language model to be used.
# 'Qwen/Qwen3.5-4B' is a 4 billion parameter model that fits on a T4 GPU.
MODEL_ID = "Qwen/Qwen3.5-4B"          # 4B open model, ~9 GB in fp16, fits a free T4. (Fallback: "Qwen/Qwen3-4B")

# Load the tokenizer associated with the specified model. The tokenizer converts text into a format
# that the model can understand (tokens).
tok = AutoTokenizer.from_pretrained(MODEL_ID)

# Load the pre-trained language model. `dtype=torch.float16` uses half-precision floating-point numbers
# to reduce memory usage, and `device_map="auto"` automatically distributes the model across available devices (e.g., GPU).
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")

In [ ]:
def chat(messages, max_new_tokens=400, temperature=0.0):
    """The one function that talks to the model: a prompt string or a list of {role, content} -> reply text."""
    # If the input messages is a single string, wrap it in the expected list format for a user message.
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    # Apply the chat template to format the messages into a single prompt string suitable for the model.
    # `tokenize=False` means it returns a string, `add_generation_prompt=True` adds a token to signal the model to start generating.
    # `enable_thinking=False` is a custom parameter for this specific model's template.
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

    # Convert the prompt string into input tensors that the model can process.
    # `return_tensors="pt"` returns PyTorch tensors, and `.to(model.device)` moves them to the model's device (e.g., GPU).
    inputs = tok(prompt, return_tensors="pt").to(model.device)

    # Generate a response from the model.
    # `max_new_tokens` limits the length of the generated reply.
    # `do_sample=temperature > 0` enables sampling (creative, varied output) if temperature is greater than 0.
    # `temperature` controls the randomness of the output; 0.0 means deterministic output.
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=temperature > 0,
                         temperature=temperature if temperature > 0 else None)

    # Decode the generated tokens back into a human-readable string.
    # `skip_special_tokens=True` removes special tokens (like start/end tokens) from the output.
    # The slice `[inputs["input_ids"].shape[1]:]` removes the input prompt from the generated output,
    # leaving only the model's new response.
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
# Call the chat function with a simple prompt and print the model's response.
# This serves as a basic test to ensure the model and chat function are working correctly.
print(chat("Say hello in five words."))

## 1: The cases and the case file

This section defines the customer messages that the agent team will process, sourced from the 'Brightwave' story. The central concept here is the **case file**, which is implemented as a simple Python dictionary. This dictionary serves as the single source of truth and communication channel for all agents. Every agent reads from and writes to this case file, recording their findings and actions. This approach ensures transparency, traceability, and avoids direct inter-agent communication, making the system easier to understand and debug. The `MESSAGES` dictionary contains various customer inquiries, each presenting a different challenge for the agent team.

In [ ]:
import time, sqlite3, pandas as pd

# Define a dictionary of customer messages. Each key is a customer's short name,
# and the value is their inquiry message. These messages will be processed by the agent team.
MESSAGES = {
"amina": """From: Amina Hassan
Subject: Credit balance refund

My account has been in credit for four months and I would like the money back rather than leaving it on the account.
I pay by direct debit and I sent a meter reading on 2 September.""",
"tom": """From: Tom Okafor
Subject: Missed smart meter appointment

Nobody came for the smart meter installation on Tuesday and no one called. I took the whole morning off work for it.
Please tell me what compensation I am owed and book a new slot.""",
"bayo": """From: Bayo Adeyemi
Subject: Complaint about waiting times

I have called four times this month and waited over forty minutes each time before giving up.
I want the goodwill payment you promised in July and never paid, and I want this treated as a formal complaint.""",
"sofia": """From: Sofia Pereira
Subject: Refund after moving out

I moved out of the flat on 14 August and gave a closing reading the same day. The final statement shows I am owed 62 pounds.
Three weeks later there is still no refund in my bank account.""",
"thanks": """From: Grace Osei
Subject: Thank you

Just a quick note to say the engineer yesterday was brilliant. Sorted the meter in twenty minutes. Thank you!""",
}

# Define a function to create a new case file for a given message.
# The case file is a dictionary that will store all information related to a customer inquiry.
def new_case(message):
    """The shared case file. Every agent reads it and writes its own part; agents never talk to each other directly."""
    # Extract the customer's name from the message using a regular expression.
    customer_name = re.search(r"^From:\s*(.+)$", message, re.M).group(1).strip()
    # Initialize the case file with basic information and empty fields for agents to populate.
    return {"message": message, "customer": customer_name,
            "kind": None, "urgency": None, "needs": [], "escalate": False, "findings": {}, "draft": None, "review": None, "rounds": 0}

# Initialize an empty list to store the trace of agent actions.
# This will be used to record who was called, when, and for how long.
TRACE = []

# Define a function to record an agent's action in the TRACE list.
def record(case, agent, note, started, calls):
    """Print what an agent did, and keep it: this trace is how the run is read afterwards."""
    # Append a dictionary with details of the agent's action to the TRACE list.
    TRACE.append({"customer": case["customer"], "agent": agent, "model calls": calls, "seconds": round(time.time() - started, 1)})
    # Print a formatted log message to the console.
    print(f"\n{'━' * 72}\n {agent.upper():<10} {case['customer']}  ·  {calls} call{'s' if calls != 1 else ''}, {time.time() - started:.0f}s\n{'━' * 72}\n{note}")

# Print the message for 'Tom Okafor' as an example.
print(MESSAGES["tom"])


## 2: What the specialists know

This section establishes the knowledge bases for our specialized agents. We have six internal Brightwave Energy policy documents, which are indexed for efficient retrieval using `chromadb` and `SentenceTransformerEmbeddingFunction`, similar to the retrieval-augmented generation (RAG) approach in Lab 1. This forms the knowledge base for the **policy specialist**.

Additionally, a SQLite database is created from a Pandas DataFrame containing ten customer records. This database is designed for the **account specialist** to query, mirroring the SQL interaction from Lab 2.

The key principle here is modularity: each specialist has a distinct, limited knowledge source and cannot access the other's information. This prevents cross-contamination and ensures that findings are directly traceable to their source, promoting accuracy and explainability within the agent team.

In [ ]:
# Define a list of dictionaries, where each dictionary represents a Brightwave Energy policy document.
# Each document has an 'id', 'region', and 'text' containing the policy details.
DOCS = [
 {"id": "refund_policy", "region": "UK", "text": """Credit balance refunds: customers who pay by direct debit can request a refund of their credit balance
at any time. Approved refunds are paid within 10 working days to the original payment method. Refunds above GBP 250 must first be
reviewed by the Billing team, who confirm an up-to-date meter reading. Duplicate payments are refunded in full within 5 working days.
Customers on the Priority Services Register are fast-tracked and paid within 5 working days."""},
 {"id": "complaints", "region": "UK", "text": """Complaints: we acknowledge every complaint within 2 working days and aim to resolve it within
10 working days. If a complaint is still open after 8 weeks, the customer may go to the Energy Ombudsman and we must tell them so in
writing. Frontline staff may offer goodwill payments up to GBP 50; larger amounts need a team leader."""},
 {"id": "smart_meter", "region": "UK", "text": """Smart meter installation: appointments are free, take 60-90 minutes and use 4-hour windows,
Monday to Saturday. Cancelling or moving an appointment is free up to 48 hours before. If our engineer misses an appointment without one
working day's notice, the customer receives GBP 30 compensation automatically within 10 working days."""},
 {"id": "leave_uk", "region": "UK", "text": """Annual leave, United Kingdom: full-time employees receive 25 days plus bank holidays. The leave year
runs January to December. Up to 5 unused days may be carried over into the next year and must be used by 31 March or they are lost."""},
 {"id": "leave_ireland", "region": "IE", "text": """Annual leave, Ireland: full-time employees in the Dublin office receive 22 days plus public
holidays. The leave year runs January to December. Up to 8 unused days may be carried over into the next year and must be used by 30 April."""},
 {"id": "escalation", "region": "ALL", "text": """Escalation rules for assistants: any mention of a gas smell or a safety risk -> give the National
Gas Emergency number 0800 111 999 and hand over to a human immediately. Refunds above GBP 250 -> hand over to Billing. Complaints open
for more than 8 weeks, legal threats, or a third contact about the same issue -> hand over to the Complaints team. Never disclose bank
details, passwords or another person's data."""},
]

# Import chromadb for creating a vector database and SentenceTransformerEmbeddingFunction for generating embeddings.
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# Initialize the Sentence Transformer embedding function. This model ('all-MiniLM-L6-v2') will convert text into numerical vectors.
# `device="cuda"` specifies that the embedding model should run on the GPU if available.
embedder = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2", device="cuda")

# Create a ChromaDB client and then create a collection named 'policies'.
# This collection will store our policy documents as embeddings.
# `embedding_function=embedder` tells ChromaDB to use our defined embedder.
# `metadata={"hnsw:space": "cosine"}` sets the distance metric for similarity searches.
policies = chromadb.Client().create_collection("policies", embedding_function=embedder, metadata={"hnsw:space": "cosine"})

# Add the policy documents to the ChromaDB collection.
# `ids` are unique identifiers for each document.
# `documents` are the text content to be embedded.
# `metadatas` are additional key-value pairs associated with each document (here, the source ID).
policies.add(ids=[d["id"] for d in DOCS], documents=[d["text"] for d in DOCS], metadatas=[{"source": d["id"]} for d in DOCS])

# Define a Pandas DataFrame containing customer information. This will act as the account specialist's knowledge base.
CUSTOMERS = pd.DataFrame([
 ("BW-1001", "Amina Hassan",     "London",     "Fixed 12",  86.40, 4, 0, 0, 1, "2025-09-02", "pays by direct debit"),
 ("BW-1002", "Tom Okafor",       "Midlands",   "Variable", -12.10, 0, 0, 0, 1, "2025-08-28", "smart meter install booked 9 Sep: engineer did not attend, no notice given"),
 ("BW-1003", "Erik Lindqvist",   "Scotland",   "Variable", -151.00, 0, 0, 0, 1, "2025-07-30", "Aug bill 74.00, Sep bill 151.00; Sep used an estimated reading"),
 ("BW-1004", "Sofia Pereira",    "London",     "Fixed 12",  61.50, 1, 0, 0, 2, "2025-08-14", "moved out 14 Aug; final statement issued 20 Aug; refund not yet paid"),
 ("BW-1005", "Claire Donnelly",  "North West", "Green Plus", -40.20, 0, 0, 0, 1, "2025-09-04", "smart meter fitted 1 Sep"),
 ("BW-1006", "Hannah Whitfield", "North West", "Fixed 24",  -33.00, 0, 0, 0, 1, "2025-09-01", "fixed tariff ends 2026-03-31"),
 ("BW-1007", "Bayo Adeyemi",     "Midlands",   "Variable",  -58.75, 0, 0, 1, 4, "2025-08-20", "goodwill payment of 40.00 promised 15 Jul, not paid"),
 ("BW-1008", "Petra Novak",      "Scotland",   "Fixed 12",  -21.00, 0, 0, 0, 2, "2025-08-30", "third-floor flat; two install visits declined by engineer"),
 ("BW-1009", "Lewis Campbell",   "Scotland",   "Variable",  12.00, 1, 1, 0, 0, "2025-09-01", "on the Priority Services Register"),
 ("BW-1010", "Grace Osei",       "London",     "Green Plus", -18.40, 0, 0, 0, 1, "2025-09-10", "smart meter fitted 10 Sep"),
], columns=["customer_id", "name", "region", "tariff", "balance_gbp", "months_in_credit", "priority_services", "open_complaints", "contacts_this_month", "last_reading", "notes"])

# Connect to an in-memory SQLite database.
conn = sqlite3.connect(":memory:")

# Write the CUSTOMERS DataFrame to a SQL table named "customers" in the SQLite database.
# `index=False` prevents Pandas from writing the DataFrame index as a column.
CUSTOMERS.to_sql("customers", conn, index=False)

# Define the schema of the customers table for the account specialist to reference.
# This string includes column names and a hint about the 'balance_gbp' column.
SCHEMA = "customers(" + ", ".join(CUSTOMERS.columns) + ")  -- balance_gbp > 0 means the customer is in credit"

# Print a summary of the loaded data: number of policies and customers.
print(len(DOCS), "policies indexed ·", len(CUSTOMERS), "customers in SQLite")

# Display the head of a selection of columns from the CUSTOMERS DataFrame for a quick preview.
CUSTOMERS[["customer_id", "name", "balance_gbp", "contacts_this_month", "notes"]].head(4)

## Step 1: Triage — the router

The Triage agent is the first point of contact for every customer message. Its primary role is to act as a router, making initial decisions about the case. It analyzes the message to determine:

*   The **kind** of case (e.g., refund, complaint, appointment).
*   Its **urgency** (low, normal, high).
*   **Which specialists it needs** (e.g., 'policy' for policy-related questions, 'account' for customer record inquiries, or 'nobody' if no specialist is required).
*   Whether it needs to **escalate** straight to a human. This decision is based on predefined escalation rules, which are derived directly from company policy (e.g., gas smell, legal threats, repeated contacts for the same issue). This ensures that the model applies rules rather than inventing them.

By handling these initial classifications and routing, the Triage agent ensures that simple messages are processed efficiently and complex ones are directed to the appropriate experts or escalated, minimizing unnecessary work for other agents.

In [ ]:
def chat_json(messages, **kw):
    """Same as chat(), but returns the first JSON object found in the reply as a dict."""
    # Call the base chat function to get a text response from the model.
    text = chat(messages, **kw)
    # Find the starting position of the first JSON object in the text.
    start = text.find("{")
    try:
        # If a JSON object is found, attempt to decode it. `.raw_decode(text[start:])` tries to decode JSON
        # from the starting point, and `[0]` gets the first decoded object.
        # Anything after the first JSON object is ignored.
        return json.JSONDecoder().raw_decode(text[start:])[0] if start >= 0 else {}   # ignore anything after the object
    except json.JSONDecodeError:
        # If a JSON decoding error occurs (e.g., truncated or malformed JSON),
        # return an empty dictionary.
        return {}                                                                      # a truncated or broken reply -> no decision

In [ ]:
def triage(case):
    """Read the message; decide what kind of case it is, who is needed, and whether a person must take it now."""
    # Record the starting time to measure the agent's processing duration.
    started = time.time()

    # Use the chat_json function to get a structured JSON response from the model.
    # The prompt instructs the model to act as a triage agent for an energy supplier,
    # specifying the available specialists ('policy', 'account') and rules for escalation.
    # It requests the output in a specific JSON format.
    reply = chat_json(f"You triage customer messages for an energy supplier. Two specialists exist: 'policy' knows what our rules allow, "
                      f"'account' can look up the customer's record. Only list a specialist if this message needs it; a thank-you needs neither.\n\n"
                      f"Escalate straight to a person only for: a gas smell or safety risk, a legal threat, or a customer whose message says this is "
                      f"their third or later contact about the same issue. Everything else, refunds and complaints included, stays with the team.\n\n"
                      f"Message:\n{case['message']}\n\n"
                      f'Return JSON: {{\"kind\": \"refund | appointment | billing | complaint | other\", \"urgency\": \"low | normal | high\", '
                      f'\"needs\": [...], \"escalate\": true/false, \"why\": \"one sentence\"}}')

    # Update the case dictionary with the information extracted from the model's JSON reply.
    # It retrieves 'kind', 'urgency', 'escalate' (converting to boolean), and 'needs' (filtering to valid specialists).
    case.update(kind=reply.get("kind"), urgency=reply.get("urgency"), escalate=str(reply.get("escalate")).lower() == "true",
                needs=[n for n in reply.get("needs", []) if n in ("policy", "account")])

    # Record the triage agent's action and findings in the global TRACE list.
    record(case, "triage", f"{case['kind']} · {case['urgency']} · needs {case['needs'] or 'nobody'}"
           + ("  →  ESCALATE: " if case["escalate"] else "  ·  ") + str(reply.get("why", "")), started, calls=1)

    # Return the updated case.
    return case

# Iterate through a selection of customer messages ('amina', 'bayo', 'thanks').
# For each message, create a new case and pass it to the triage agent to process.
for key in ("amina", "bayo", "thanks"):
    triage(new_case(MESSAGES[key]))

## Step 2: The specialists

This section defines the two core specialist agents: the **policy specialist** and the **account specialist**. Each is designed to be highly focused, with a single job, a dedicated knowledge source, and a concise prompt, embodying the principle of

In [ ]:
def policy_specialist(case, question=None):
    """What do our rules allow for this case? Two passages in, a short cited finding out."""
    # Record the starting time to measure the agent's processing duration.
    started = time.time()

    # Query the ChromaDB 'policies' collection for relevant policy documents.
    # The query text is either a specific `question` (if provided by the reviewer) or the original `case["message"]`.
    # `n_results=2` retrieves the top 2 most relevant policy passages.
    r = policies.query(query_texts=[question or case["message"]], n_results=2)

    # Format the retrieved policy passages and their sources into a single string.
    passages = "\n\n".join(f"[{m['source']}] {d}" for d, m in zip(r["documents"][0], r["metadatas"][0]))

    # Use the chat function to generate a finding based on the policy passages and the customer message.
    # The prompt instructs the model to act as a policy specialist, state what the policy allows/requires,
    # quote amounts/timeframes, and cite sources. It also handles specific questions from a reviewer.
    finding = chat(f"Policy passages:\n{passages}\n\nCustomer message:\n{case['message']}\n\n"
                   + (f"Specific question from the reviewer: {question}\n\n" if question else "")
                   + f"In 2-4 short sentences, state exactly what the policy allows or requires for this case, quoting every amount and timeframe, "
                   f"with the source id in square brackets after each fact. If the passages do not cover it, say so plainly.", max_new_tokens=220)

    # Store the generated policy finding in the case file under 'findings'.
    case["findings"]["policy"] = finding

    # Record the policy specialist's action and finding in the global TRACE list.
    record(case, "policy", finding, started, calls=1)

    # Return the updated case.
    return case

def account_specialist(case, question=None):
    """What does the customer's record show? One SQL query, repaired once if it fails."""
    # Record the starting time and initialize call count, SQL query, and error variables.
    started, calls, sql, error = time.time(), 0, None, None

    # Determine the question for the SQL query. It's either a specific `question` from the reviewer
    # or a general request to find relevant information from the customer's message.
    ask = question or "everything on this customer's record that matters for their message: " + case["message"].splitlines()[1]

    # Loop up to two attempts to generate and execute an SQL query.
    for attempt in range(2):
        # Use the chat function to generate an SQL query. The prompt provides the database schema,
        # customer name, and the question. It also includes feedback if a previous query failed.
        reply = chat(f"Write one SQLite SELECT over the table {SCHEMA}.\nThe customer is {case['customer']!r}.\nQuestion: {ask}\n"
                     + (f"\nYour previous query:\n{sql}\nfailed with: {error}\nWrite a corrected query.\n" if error else "")
                     + "Reply with the SQL in a ```sql block.", max_new_tokens=150); calls += 1

        # Extract the SQL query from the model's reply, looking for text within a ```sql block.
        block = re.search(r"```sql\s*(.*?)```", reply, re.S | re.I)
        sql = (block.group(1) if block else reply).strip().rstrip(";")

        try:
            # Execute the SQL query against the in-memory SQLite database.
            rows = pd.read_sql(sql, conn); break
        except Exception as e:
            # If the query fails, store the error message and set `rows` to None.
            error, rows = str(e).splitlines()[-1], None

    # Format the finding: if rows were returned, convert them to a string; otherwise, report the error.
    finding = rows.to_string(index=False) if rows is not None and len(rows) else f"could not read the account ({error or 'no matching record'})"

    # Store the generated account finding in the case file under 'findings'.
    case["findings"]["account"] = finding

    # Record the account specialist's action and finding (including the SQL query) in the global TRACE list.
    record(case, "account", f"{sql}\n\n{finding}", started, calls=calls)

    # Return the updated case.
    return case

# Define a dictionary mapping specialist names to their respective functions.
SPECIALISTS = {"policy": policy_specialist, "account": account_specialist}

# Create a new case for Amina's message and pass it to the triage agent.
case = triage(new_case(MESSAGES["amina"]))

# Iterate through the specialists identified as 'needed' by the triage agent.
# For each needed specialist, call their respective function to process the case.
for name in case["needs"]:
    case = SPECIALISTS[name](case)

## Step 3: The writer

The writer agent's role is to synthesize the information gathered by the specialists and the original customer message into a coherent draft reply. It operates under strict constraints:

*   It must use **only** facts, amounts, and timeframes explicitly found in the specialists' findings.
*   It is explicitly forbidden from introducing information from its own memory or making unverified claims.
*   It needs to acknowledge the issue, state what will happen next, and provide a timeframe.
*   The draft should be concise (under 120 words), plain, and warm in tone.
*   It must sign off as 'Brightwave Customer Team.'

This deliberate limitation on the writer's knowledge and its reliance on the case file sets up the critical function of the next agent: the reviewer. The writer's potential to

In [ ]:
def writer(case, reasons=None):
    """Draft the reply from the case file. On a second round it also sees why the reviewer returned the last one."""
    # Record the starting time to measure the agent's processing duration.
    started = time.time()

    # Consolidate all specialist findings from the case file into a single string.
    # If no findings exist, a default message is used.
    findings = "\n\n".join(f"{name.upper()} SPECIALIST FOUND:\n{text}" for name, text in case["findings"].items()) or "No specialist findings."

    # Use the chat function to generate a draft reply based on the customer message and specialist findings.
    # The prompt specifies strict rules for the writer: use only facts from findings, acknowledge issue,
    # state next steps, keep it under 120 words, maintain a plain/warm tone, and sign off correctly.
    # If `reasons` are provided (from a previous review), the prompt includes instructions to fix the draft.
    draft = chat(f"Write a reply to this customer on behalf of Brightwave Energy.\n\nCustomer message:\n{case['message']}\n\n{findings}\n\n"
                 f"Rules: use only facts, amounts and timeframes that appear in the specialists' findings, and nothing from memory; "
                 f"acknowledge the issue, say exactly what happens next and by when; under 120 words; plain and warm; "
                 f"sign off as 'Brightwave Customer Team'."
                 + (f"\n\nThe reviewer returned your previous draft:\n{case['draft']}\n\nbecause:\n" + "\n".join("- " + r for r in reasons)
                    + "\n\nFix every point." if reasons else ""), max_new_tokens=260)

    # Store the generated draft in the case file.
    case["draft"] = draft

    # Record the writer's action and the generated draft in the global TRACE list.
    record(case, "writer", draft, started, calls=1)

    # Return the updated case.
    return case

# Call the writer function to generate an initial draft for the current case.
case = writer(case)

## Step 4: The reviewer — a critic who is not the author

The reviewer agent plays a crucial role in maintaining quality and adherence to policy. Its primary function is to either approve the writer's draft or return it with specific reasons for revision. The reviewer's evaluation involves a dual approach:

1.  **Code-based Checks**: A set of programmatic checks verifies factual accuracy and compliance. This includes:
    *   Ensuring that every monetary amount and timeframe mentioned in the draft is explicitly present in the specialists' findings.
    *   Checking for

In [ ]:
# Define a list of banned phrases that the writer should not use in the draft reply.
BANNED = ["guarantee", "immediately", "straight away", "as a gesture we will", "we will definitely"]

def reviewer(case):
    """Approve the draft, or return it with reasons. Code checks the facts, the model checks the sense."""
    # Record the starting time to measure the agent's processing duration.
    started = time.time()

    # Get the current draft and concatenate all specialist findings into a single string for evidence.
    draft, evidence = case["draft"], " ".join(case["findings"].values())

    # Extract all monetary amounts (£X or X pounds/GBP) from the draft.
    # Convert them to a set of floats for comparison.
    amounts = {f"{float(n.replace(',', '')):g}" for n in re.findall(r"£\s?(\d[\d,]*(?:\.\d+)?)", draft)
               + re.findall(r"\b(\d+(?:\.\d+)?)\s*(?:pounds|GBP)\b", draft)}                                  # money the draft promises
    # Extract all numbers from the evidence (specialist findings).
    # Convert them to a set of floats for comparison.
    known = {f"{float(n.replace(',', '')):g}" for n in re.findall(r"\d[\d,]*(?:\.\d+)?", evidence)}         # every number a specialist reported

    # Initialize a list to store any problems found with the draft.
    problems = [f"£{n} is not in any specialist's findings" for n in sorted(amounts - known)]

    # Check for timeframes (e.g., "X days") in the draft.
    # If a number or unit from the timeframe is not present in the evidence, add it as a problem.
    for span, n, unit in re.findall(r"\b((\d+)\s*(?:working\s+)?(day|hour|week|month)s?)\b", draft):                 # timeframes the draft states
        if f"{float(n):g}" not in known or unit not in evidence.lower():
            problems.append(f"'{span}' is not in any specialist's findings")

    # Check for any banned phrases in the draft and add them as problems.
    problems += [f"promises '{p}'" for p in BANNED if p in draft.lower()]

    # Check if the draft exceeds the word limit and add it as a problem if it does.
    if len(draft.split()) > 150:
        problems.append(f"{len(draft.split())} words, the limit is 150")

    # Use the chat_json function to get the model's review of the draft.
    # The prompt instructs the model to act as a reviewer, checking if the draft answers the question,
    # maintains tone, and is supported by findings. It also asks for reasons and if a specialist needs to be re-consulted.
    verdict = chat_json(f"You review replies before they reach customers.\n\nCustomer message:\n{case['message']}\n\nWhat the specialists found:\n{evidence}\n\n"
                        f"Draft reply:\n{draft}\n\nDoes the draft answer what the customer actually asked, in a plain and warm tone, without stating "
                        f"anything the findings do not support? If a needed fact is missing, name the specialist to ask ('policy' or 'account') and the question.\n"
                        f'Return JSON: {{\"approved\": true/false, \"reasons\": ["..."], \"ask_again\": [], \"question\": ""}}', max_new_tokens=250)

    # Determine final approval status: approved only if no programmatic problems and model approves.
    approved = not problems and str(verdict.get("approved")).lower() == "true"

    # Compile all reasons for returning the draft: programmatic problems + model's reasons.
    reasons = [] if approved else problems + [str(r) for r in verdict.get("reasons", []) if r]

    # Store the review results in the case file.
    # Filter 'ask_again' specialists to ensure they are valid.
    case["review"] = {"approved": approved, "reasons": reasons, "question": str(verdict.get("question") or ""),
                      "ask_again": [a for a in (verdict.get("ask_again") or []) if a in SPECIALISTS]}

    # Record the reviewer's action (APPROVED or RETURNED with reasons) in the global TRACE list.
    record(case, "reviewer", "APPROVED" if approved else "RETURNED\n" + "\n".join("- " + r for r in reasons)
           + (f"\n→ ask {case['review']['ask_again']}: {case['review']['question']}" if case["review"]["ask_again"] else ""), started, calls=1)

    # Return the updated case.
    return case

# Call the reviewer function to evaluate the current draft.
case = reviewer(case)

## Step 5: The supervisor — order, budget, exit

The supervisor agent acts as the orchestrator of the entire team workflow. Unlike the specialists or the writer/reviewer, the supervisor doesn't directly generate content or find facts; instead, it manages the control flow, budget, and exit conditions for each case. Its key responsibilities include:

*   **Ordering of Operations**: It ensures agents are called in the correct sequence: triage first, then specialists (as requested by triage or reviewer), then the writer, and finally the reviewer.
*   **Budget Management**: It enforces a limit on the number of rounds for the writer-reviewer loop (e.g., at most two rounds) to prevent infinite cycles and manage operational costs.
*   **Dynamic Dispatch**: Based on the reviewer's verdict, it can send a specialist back to work with a specific question, demonstrating an adaptive workflow.
*   **Exit Conditions**: It defines when a case should be handed over to a human. This occurs if triage initially flags an escalation, or if the reviewer refuses a draft more than a predefined number of times (e.g., twice).

This architecture is crucial for making the system robust and efficient, ensuring that cases are processed within reasonable limits and human intervention occurs when automated processing is insufficient.

In [ ]:
RESULTS = {}                                                                       # customer -> how the team's run ended

def hand_to_human(case, why):
    # Record the outcome for the customer as not approved.
    RESULTS[case["customer"]] = {"approved": False, "reasons": len(case["review"]["reasons"]) if case["review"] else 0}
    # Print a clear message indicating that the case is being escalated to a human, along with the reason.
    print(f"\n{'#' * 72}\n TO A PERSON: {case['customer']} — {why}\n{'#' * 72}")
    # Print a summary of the case's current state for debugging/logging purposes.
    print(f"kind: {case['kind']} · urgency: {case['urgency']}\nfindings: {list(case['findings'])}\nlast draft: {(case['draft'] or 'none')[:300]}"
          f"\nreviewer said: {case['review']['reasons'] if case['review'] else '-'}")
    return case

def handle(message, max_rounds=2):
    """The supervisor: triage, dispatch, write, review, loop back, or hand over."""
    # Step 1: Triage the incoming message to categorize it and identify initial needs.
    case = triage(new_case(message))

    # Check if triage has flagged the case for immediate escalation.
    if case["escalate"]:
        return hand_to_human(case, "triage flagged it")

    # Step 2: Dispatch to specialists based on triage's recommendations.
    for name in case["needs"]:
        case = SPECIALISTS[name](case)

    # Initialize reasons for rejection to None for the first writer round.
    reasons = None

    # Step 3 & 4: Writer-Reviewer loop. Attempt to generate and review a draft up to `max_rounds` times.
    for round_no in range(1, max_rounds + 1):
        case["rounds"] = round_no
        # First, the writer generates a draft (or revises it), then the reviewer evaluates it.
        case = reviewer(writer(case, reasons))

        # If the draft is approved, the process is complete.
        if case["review"]["approved"]:
            RESULTS[case["customer"]] = {"approved": True, "reasons": 0}
            # Print approval message and the final approved draft.
            print(f"\n{'#' * 72}\n APPROVED for {case['customer']} after {round_no} round{'s' if round_no > 1 else ''}\n{'#' * 72}\n{case['draft']}")
            return case

        # If not approved, store the reasons for the next writer iteration.
        reasons = case["review"]["reasons"]

        # If the reviewer requested a specialist to be re-consulted, dispatch to that specialist.
        for name in case["review"]["ask_again"]:
            case = SPECIALISTS[name](case, question=case["review"]["question"])

    # If the loop completes without approval, escalate to a human.
    return hand_to_human(case, f"the reviewer refused {max_rounds} times")

# Clear the global TRACE list before running new cases.
TRACE.clear()

# Process Tom's message using the `handle` (supervisor) function.
case = handle(MESSAGES["tom"])

Read the trace top to bottom: who was called, in what order, what each one cost. Now the other two cases. Bayo's message says he has contacted us four times, and the policy says a third contact goes to the complaints team, so watch triage take the exit before any specialist spends a call.

This section serves as a prompt for you to analyze the execution trace generated by the `handle` function. By examining the `TRACE` list and the aggregated DataFrame, you can understand:

*   The sequence of agents involved for each customer (`Tom Okafor`, `Amina Hassan`, `Bayo Adeyemi`).
*   How many model calls each agent made.
*   The time spent by each agent.

Pay close attention to `Bayo Adeyemi`'s case. The message explicitly mentions multiple contacts, which, according to the `escalation` policy document, should trigger an immediate escalation to a human. Observe how the `triage` agent identifies this and bypasses further specialist or writer/reviewer steps, directly routing the case to a human intervention point.

In [ ]:
# Process Amina's message using the `handle` (supervisor) function.
handle(MESSAGES["amina"])

# Process Bayo's message using the `handle` (supervisor) function.
handle(MESSAGES["bayo"])

# Create a Pandas DataFrame from the TRACE list.
# Group the DataFrame by customer and agent, then sum the 'model calls' and 'seconds' for each.
# This provides a summary of agent activity and cost for each customer's case.
pd.DataFrame(TRACE).groupby(["customer", "agent"], sort=False)[["model calls", "seconds"]].sum()

## The same team as a graph — LangGraph

This section introduces **LangGraph**, a library designed to build stateful, multi-agent applications. The entire workflow orchestrated by the supervisor (triage, specialists, writer, reviewer, looping, and exits) can be represented as a graph. LangGraph takes the shared `Case` file as its state, and the six agent functions (triage, specialists, writer, reviewer, done, human) become the nodes of this graph. The supervisor's decisions on routing and looping are then defined as edges and conditional edges within the graph.

What you gain by using LangGraph is that the complex control flow, which was previously embedded in the `handle()` function's `if`-statements and `for`-loops, now becomes explicit data. This offers several advantages:

*   **Visualization**: The graph can be easily drawn and understood, providing a clear visual representation of the agent team's workflow.
*   **Checkpointing**: The state of the case can be saved after each node, allowing for easy debugging, recovery, or even human intervention at any point.
*   **Resumability**: Workflows can be paused and resumed, which is especially useful for long-running processes or human-in-the-loop scenarios.
*   **Collaboration**: The explicit graph structure makes it easier for different team members to understand and contribute to the agent system without needing to deeply parse complex procedural code.

While it adds a dependency and an abstraction layer, understanding how to build the system procedurally first, as you did with the `handle()` function, demystifies the framework and highlights its benefits for managing complex agent interactions.

In [ ]:
# Install the langgraph library. `-q` for quiet installation, `"langgraph>=0.6"` for a specific version.
!pip install -q "langgraph>=0.6"

# Import necessary components from the typing module and langgraph.
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Define the `Case` state for LangGraph. TypedDict allows for type hinting,
# making the structure of the case file explicit. `total=False` means keys are optional.
class Case(TypedDict, total=False):                                          # the case file, declared as the graph's state
    message: str
    customer: str
    kind: str
    urgency: str
    needs: list
    escalate: bool
    findings: dict
    draft: str
    review: dict
    rounds: int

# Define a node function for specialists.
def specialists_node(case):
    """Run the specialists triage asked for, or the one the reviewer sent back with a question."""
    # Determine if the reviewer has sent the case back to a specialist with a specific question.
    sent_back = bool(case.get("review")) and bool(case["review"]["ask_again"])

    # Iterate through the specialists needed (either from initial triage or reviewer's request).
    for name in (case["review"]["ask_again"] if sent_back else case["needs"]):
        # Call the relevant specialist function, passing the specific question if sent back by the reviewer.
        case = SPECIALISTS[name](case, question=case["review"]["question"] if sent_back else None)
    return case

# Define a node function for the writer.
def writer_node(case):
    # Increment the round counter for the writer-reviewer loop.
    case["rounds"] = case.get("rounds", 0) + 1
    # Call the writer function, passing reasons for revision if the case was previously reviewed.
    return writer(case, case["review"]["reasons"] if case.get("review") else None)

# Define a node function for when the case is successfully completed.
def done_node(case):
    # Mark the case as approved in the RESULTS dictionary.
    RESULTS[case["customer"]] = {"approved": True, "reasons": 0}
    # Print an approval message including the final draft.
    print(f"\n{'#' * 72}\n APPROVED for {case['customer']} after {case['rounds']} round(s)\n{'#' * 72}\n{case['draft']}")
    return case

# Define a routing function to decide the next step after triage.
def after_triage(case):
    # If the case is escalated by triage, go to 'human'; otherwise, go to 'specialists'.
    return "human" if case["escalate"] else "specialists"

# Define a routing function to decide the next step after review.
def after_review(case, max_rounds=2):                                        # the supervisor's if-statements, as a routing function
    # If the draft is approved, the process is 'done'.
    if case["review"]["approved"]:
        return "done"
    # If the maximum number of review rounds has been reached, escalate to 'human'.
    if case["rounds"] >= max_rounds:
        return "human"
    # If the reviewer asked a specialist to be re-consulted, go back to 'specialists'; otherwise, go back to 'writer'.
    return "specialists" if case["review"]["ask_again"] else "writer"

# Initialize the StateGraph with our `Case` state type.
graph = StateGraph(Case)

# Add nodes to the graph, mapping names to their respective functions.
for name, fn in [("triage", triage), ("specialists", specialists_node), ("writer", writer_node), ("reviewer", reviewer),
                 ("done", done_node), ("human", lambda case: hand_to_human(case, "escalated, or the reviewer refused twice"))]:
    graph.add_node(name, fn)

# Define the edges (transitions) between nodes.
# Start the graph at the 'triage' node.
graph.add_edge(START, "triage")

# Add conditional edges from 'triage' based on the `after_triage` function's return value.
graph.add_conditional_edges("triage", after_triage, {"human": "human", "specialists": "specialists"})

# Define a direct edge from 'specialists' to 'writer'.
graph.add_edge("specialists", "writer")

# Define a direct edge from 'writer' to 'reviewer'.
graph.add_edge("writer", "reviewer")

# Add conditional edges from 'reviewer' based on the `after_review` function's return value.
graph.add_conditional_edges("reviewer", after_review, {"done": "done", "human": "human", "writer": "writer", "specialists": "specialists"})

# Define direct edges to mark the end of the graph for 'done' and 'human' paths.
graph.add_edge("done", END)
graph.add_edge("human", END)

# Compile the graph into an executable team agent.
team = graph.compile()

In [ ]:
# Import necessary modules for displaying images.
from IPython.display import Image, display

try:
    # Attempt to draw and display a Mermaid diagram of the LangGraph team workflow as a PNG image.
    # This provides a visual representation of the agent interactions and their flow.
    display(Image(team.get_graph().draw_mermaid_png()))                        # drawn by the library from the edges above
except Exception:
    # If displaying the PNG fails (e.g., due to missing renderer),
    # print the Mermaid diagram as plain text instead.
    print(team.get_graph().draw_mermaid())                                     # the same diagram as text, if the renderer is unreachable

In [ ]:
# Clear the global TRACE list to start fresh for the LangGraph execution.
TRACE.clear()

# Invoke the LangGraph team agent with Amina's message.
# This runs the entire workflow defined by the graph and returns the final state of the case.
# The result (`final`) will contain the outcome of processing Amina's message.
final = team.invoke(new_case(MESSAGES["amina"]))                              # same agents, same trace, same cost

## Exercises & Recap

This section provides opportunities to further experiment with and reflect on the agent team architecture. It also summarizes the distinct responsibilities of each agent, reinforcing the concepts learned in this lab.

1.  **Parallel specialists.** The policy and account specialists do not depend on each other. Run them concurrently and measure the seconds column.
    *   **Challenge**: Modify the `specialists_node` (or the `handle` function if not using LangGraph) to run the `policy_specialist` and `account_specialist` in parallel when both are needed. You might use Python's `concurrent.futures` module (e.g., `ThreadPoolExecutor`) to achieve this. Observe the impact on the `seconds` column in the `TRACE` output – it should reduce the total time spent by specialists for cases requiring both.

2.  **A second reviewer with a different checklist**, for tone only. Do two critics agree? When they disagree, who wins?
    *   **Challenge**: Create a new agent, for example, `tone_reviewer`, which focuses solely on evaluating the tone of the draft reply. Integrate this new reviewer into the workflow. Consider how to handle situations where the original reviewer and the new tone reviewer disagree on whether a draft should be approved. You might introduce a voting mechanism or define a hierarchy (e.g., the original reviewer's decision takes precedence).

| Agent | Decides | Reuses |
|---|---|---|
| Triage | kind, urgency, who is needed, escalate | the router pattern from your Day 4 deck |
| Policy specialist | what the rules allow, cited | Lab 1's retrieval |
| Account specialist | what the record shows | Lab 2's SQL with repair |
| Writer | the reply, from findings only | |
| Reviewer | approve or return, with a specialist to ask again | code checks the facts, the model checks the sense |
| Supervisor | order, budget, exit to a person | the Interactive lab's approval idea, as an exit |
| The supervisor, drawn | the same six functions as nodes, the verdicts as edges | LangGraph `StateGraph` |